[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/takeshun1984/NumeralAnalysisInGeophysics_SolidEarth/blob/main/08_FDM_2nd_homo_P-SV_CerjanZB.ipynb)

In [ ]:
import numpy as np
import os

# 領域設定など
SP = np.float32
NX, NZ = 400, 400
DX, DZ = 0.4, 0.4
DT = 0.02
NTMAX = 2000

# 震源情報
I0, K0 = NX // 2, NZ // 2
T0, TS = 5.0, 0.0
MXX, MZZ, MXZ, MO = 0.0, 0.0, 1.0, 1.0
DTXZ = DT / (DX * DZ)

# --- 出力・減衰設定 ---
NTDEC, NXD, NZD = 50, 2, 2
DST = 10.0
NST = int(NX * DX / DST) - 1
NSKIP = 2
NWMAX = NTMAX // NSKIP
OUTDIR = "output/08"  # ノートブックごとに出力先を分ける
ONAME0, WNAME0 = f"{OUTDIR}/psv.h.", f"{OUTDIR}/wav.h."

# --- 関数定義 ---
def kupper(t, ts, tr):
    if ts <= t <= ts + tr:
        return 3 * np.pi * (np.sin(np.pi * (t - ts) / tr))**3 / (4 * tr)
    else:
        return 0.0

# 配列の初期化 0~NX+1
SXX = np.zeros((NX + 2, NZ + 2), dtype=SP)
SZZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
SXZ = np.zeros((NX + 2, NZ + 2), dtype=SP)
VX  = np.zeros((NX + 2, NZ + 2), dtype=SP)
VZ  = np.zeros((NX + 2, NZ + 2), dtype=SP)

# 均質媒質
VP, VS, RO = 6.0, 3.5, 2.3
RIG_val = RO * VS**2
LAM_val = RO * VP**2 - 2.0 * RO * VS**2

# 各格子点に物性を割り当て
RIG = np.full((NX + 2, NZ + 2), RIG_val, dtype=SP)
LAM = np.full((NX + 2, NZ + 2), LAM_val, dtype=SP)
RHO = np.full((NX + 2, NZ + 2), RO, dtype=SP)

# 波形記録用配列
VWX = np.zeros((NWMAX, NST), dtype=SP)
VWZ = np.zeros((NWMAX, NST), dtype=SP)
ISTX = np.array([int((ist+1)*DST/DX) for ist in range(NST)])
ISTZ = NZ // 2

# --- 吸収境界条件 (Cerjan 1985) パラメータ ---
BETA, NA = 0.09, 20

# --- 吸収境界係数の計算 ---
gx1, gx2 = np.ones(NX+2, dtype=SP), np.ones(NX+2, dtype=SP)
gz1, gz2 = np.ones(NZ+2, dtype=SP), np.ones(NZ+2, dtype=SP)
for i in range(1, NA + 1):
    val1 = np.exp(-BETA*(1.0-(i-0.5)/NA)**2)
    val2 = np.exp(-BETA*(1.0-i/NA)**2)
    
    gx1[i], gz1[i], gx2[i], gz2[i] = val1, val1, val2, val2

    gx1[NX-i+1], gx2[NX-i+1], gz1[NZ-i+1], gz2[NZ-i+1] = val2, val1, val2, val1

GX1, GZ1 = np.meshgrid(gx1, gz1, indexing='ij')
GX2, GZ2 = np.meshgrid(gx2, gz2, indexing='ij')

以下で、Qのモデル設定を実施している。
Zener bodyを用いるため緩和時間に関する部分と、メモリ変数の初期化を含む。

In [ ]:
F0, Q0 = 1.0 / T0, 10.0

QS, QP = np.full((NX+2, NZ+2), 1.0e6, dtype=SP), np.full((NX+2, NZ+2), 1.0e6, dtype=SP)
QS[0:NX//2-1, :] = Q0
QP[0:NX//2-1, :] = 2.0 * Q0

W0   = 2.0 * np.pi * F0
TAU  = 1.0 / W0 * (np.sqrt(1.0 + QP**(-2)) - 1.0 / QP)
TAUP_ratio = 1.0 / (W0**2 * TAU**2)
TAUS_ratio = (1.0 + W0 * TAU * QS) / (W0 * QS * TAU - (W0 * TAU)**2)
TU = -1.0 / TAU

RXX = np.zeros((NX+2, NZ+2), dtype=SP)
RZZ = np.zeros((NX+2, NZ+2), dtype=SP)
RXZ = np.zeros((NX+2, NZ+2), dtype=SP)

### 係数の事前計算（高速化）

物性（$\lambda, \mu, \rho, Q$）は時間変化しないため、ループ内で毎ステップ同じ係数を計算し直すのは無駄である。そこで、格子点ごとの係数をループの前に1回だけ計算しておく。係数は格子点ごとの配列なので、不均質媒質でもそのまま使える。

**メモリ変数**：$\tau_\sigma$を応力緩和時間（コードでは`TAU`、`TU`$=-1/\tau_\sigma$）とすると、メモリ変数$r_{xx}$は
$$
\frac{\partial r_{xx}}{\partial t} = -\frac{1}{\tau_\sigma}\left[r_{xx} + F_{xx}\right]
$$
に従う。ここで
$$
F_{xx} = (\lambda+2\mu)\left(\frac{\tau_\epsilon^P}{\tau_\sigma}-1\right)(\partial_x v_x+\partial_z v_z) - 2\mu\left(\frac{\tau_\epsilon^S}{\tau_\sigma}-1\right)\partial_z v_z
$$
である。$r_{zz}$、$r_{xz}$も同じ形の式に従い、
$$
F_{zz} = (\lambda+2\mu)\left(\frac{\tau_\epsilon^P}{\tau_\sigma}-1\right)(\partial_x v_x+\partial_z v_z) - 2\mu\left(\frac{\tau_\epsilon^S}{\tau_\sigma}-1\right)\partial_x v_x, \quad
F_{xz} = \mu\left(\frac{\tau_\epsilon^S}{\tau_\sigma}-1\right)(\partial_x v_z+\partial_z v_x)
$$
である。これをCrank–Nicolson法で離散化すると
$$
\frac{r_{xx}^{n+1}-r_{xx}^{n}}{\Delta t} = -\frac{1}{\tau_\sigma}\left[\frac{r_{xx}^{n+1}+r_{xx}^{n}}{2} + F_{xx}\right]
$$
となり、これを$r_{xx}^{n+1}$について解くと
$$
r_{xx}^{n+1} = C_1\, r_{xx}^{n} + C_2\, F_{xx}, \quad
C_1 = \frac{1-\Delta t/(2\tau_\sigma)}{1+\Delta t/(2\tau_\sigma)}, \quad
C_2 = \frac{-\Delta t/\tau_\sigma}{1+\Delta t/(2\tau_\sigma)}
$$
となる。そこで、`P_DIV`$=C_2(\lambda+2\mu)(\tau_\epsilon^P/\tau_\sigma-1)$、`P_2MU`$=C_2\,2\mu(\tau_\epsilon^S/\tau_\sigma-1)$、`P_MU`$=C_2\,\mu(\tau_\epsilon^S/\tau_\sigma-1)$のように、$F$の各項の係数に$C_2$まで掛けたものを用意する。

**応力**：
$$
\sigma_{xx}^{n+1} = \sigma_{xx}^{n} + \Delta t\left[(\lambda+2\mu)\frac{\tau_\epsilon^P}{\tau_\sigma}(\partial_x v_x+\partial_z v_z) - 2\mu\frac{\tau_\epsilon^S}{\tau_\sigma}\partial_z v_z + \frac{r_{xx}^{n+1}+r_{xx}^{n}}{2}\right]
$$
の係数に$\Delta t$まで掛けたものを`S_DIV`、`S_2MU`とする（$\sigma_{zz}$、$\sigma_{xz}$も同様）。

**速度**：$\Delta t/\rho$を`BX_DT`、`BZ_DT`とし、Cerjanの吸収境界の係数の積（`GX1*GZ1`など）もあらかじめ計算しておく。また、`/DX`は`*rDX`（$1/\Delta x$）として割り算を掛け算に置き換える。


In [ ]:
# ---- 時間ループで使う係数をあらかじめ計算（物性は時間変化しないので1回だけ） ----
IN = (slice(1, NX+1), slice(1, NZ+1))   # 更新する内部領域 [1:NX+1, 1:NZ+1]
rDX, rDZ = SP(1.0/DX), SP(1.0/DZ)        # 割り算を掛け算に
HDT = SP(0.5*DT)

# SXZ の位置の剛性と、VX, VZ の位置の 1/ρ（均質媒質なので格子点の値そのまま）
RIG_XZ = RIG[IN]
BX = BZ = 1.0/RHO[IN]

PI_  = (LAM + 2.0*RIG)[IN]                # λ+2μ
MU_  = RIG[IN]                            # μ（SXX, SZZ の位置）
TP, TS_ = TAUP_ratio[IN], TAUS_ratio[IN]  # τεP/τσ, τεS/τσ
TU_  = TU[IN]                             # -1/τσ

# メモリ変数 (Crank-Nicolson): R_new = C1*R_old + C2*F
C1 = (1.0 + TU_*DT*0.5) / (1.0 - TU_*DT*0.5)
C2 = TU_*DT / (1.0 - TU_*DT*0.5)
P_DIV = C2 * PI_*(TP-1.0)                 # F の各項の係数に C2 まで掛けたもの
P_2MU = C2 * 2.0*MU_*(TS_-1.0)
P_MU  = C2 * RIG_XZ*(TS_-1.0)

# 応力: 各項の係数に Δt まで掛けたもの
S_DIV = PI_*TP*DT
S_2MU = 2.0*MU_*TS_*DT
S_MU  = RIG_XZ*TS_*DT

# 速度: Δt/ρ
BX_DT, BZ_DT = BX*DT, BZ*DT

# 単精度にそろえる
C1, P_DIV, P_2MU, P_MU, S_DIV, S_2MU, S_MU, BX_DT, BZ_DT = (
    a.astype(SP) for a in (C1, P_DIV, P_2MU, P_MU, S_DIV, S_2MU, S_MU, BX_DT, BZ_DT))

# Cerjan 吸収境界の係数の積
G_S, G_SXZ = GX1*GZ1, GX2*GZ2
G_VX, G_VZ = GX2*GZ1, GX1*GZ2


以下でメインループを回しているが、
```
    # 2. メモリ変数更新: R_new = C1*R_old + C2*F
    RXXN, RZZN, RXZN = RXX[IN].copy(), RZZ[IN].copy(), RXZ[IN].copy()
    RXX[IN] = C1*RXXN + P_DIV*div - P_2MU*dzvz
    RZZ[IN] = C1*RZZN + P_DIV*div - P_2MU*dxvx
    RXZ[IN] = C1*RXZN + P_MU*shear

    # 3. 応力場更新
    SXX[IN] += S_DIV*div - S_2MU*dzvz + HDT*(RXX[IN] + RXXN)
    SZZ[IN] += S_DIV*div - S_2MU*dxvx + HDT*(RZZ[IN] + RZZN)
    SXZ[IN] += S_MU*shear             + HDT*(RXZ[IN] + RXZN)
```
とこれまでの応力のアップデートと比べて、前のメモリ変数の保存、メモリ変数の更新、応力の更新と段階が3つに別れている。`IN`は内部領域`[1:NX+1, 1:NZ+1]`を表し、係数（`C1`、`P_DIV`、`S_DIV`など）は上のセルで事前に計算したものである。


In [ ]:
# メインループ
print(f"{'Step':>5} / {NTMAX}: {'Time':>7} {'Vxmax':>12}")
os.makedirs(OUTDIR, exist_ok=True)

# 再実行しても前回の結果の続きにならないよう、波動場を初期化
for arr in (SXX, SZZ, SXZ, VX, VZ, RXX, RZZ, RXZ, VWX, VWZ):
    arr[:] = 0.0

for it in range(1, NTMAX + 1):
    T = it * DT

    # 1. 速度の空間微分
    dxvx = (VX[1:NX+1, 1:NZ+1] - VX[0:NX  , 1:NZ+1]) * rDX
    dzvz = (VZ[1:NX+1, 1:NZ+1] - VZ[1:NX+1, 0:NZ  ]) * rDZ
    dxvz = (VZ[2:NX+2, 1:NZ+1] - VZ[1:NX+1, 1:NZ+1]) * rDX
    dzvx = (VX[1:NX+1, 2:NZ+2] - VX[1:NX+1, 1:NZ+1]) * rDZ
    div   = dxvx + dzvz   # ∂x vx + ∂z vz
    shear = dxvz + dzvx   # ∂x vz + ∂z vx

    # 2. メモリ変数更新: R_new = C1*R_old + C2*F
    RXXN, RZZN, RXZN = RXX[IN].copy(), RZZ[IN].copy(), RXZ[IN].copy()
    RXX[IN] = C1*RXXN + P_DIV*div - P_2MU*dzvz
    RZZ[IN] = C1*RZZN + P_DIV*div - P_2MU*dxvx
    RXZ[IN] = C1*RXZN + P_MU*shear

    # 3. 応力場更新
    SXX[IN] += S_DIV*div - S_2MU*dzvz + HDT*(RXX[IN] + RXXN)
    SZZ[IN] += S_DIV*div - S_2MU*dxvx + HDT*(RZZ[IN] + RZZN)
    SXZ[IN] += S_MU*shear             + HDT*(RXZ[IN] + RXZN)

    SXX *= G_S
    SZZ *= G_S
    SXZ *= G_SXZ

    # 震源注入
    sdrop = MO * kupper(T, TS, T0) * DTXZ
    SXX[I0, K0] -= MXX * sdrop
    SZZ[I0, K0] -= MZZ * sdrop
    SXZ[I0-1:I0+1, K0-1:K0+1] -= MXZ * sdrop * 0.25

    # 4. 速度場の更新
    dxsxx = (SXX[2:NX+2, 1:NZ+1] - SXX[1:NX+1, 1:NZ+1]) * rDX
    dxsxz = (SXZ[1:NX+1, 1:NZ+1] - SXZ[0:NX  , 1:NZ+1]) * rDX
    dzszz = (SZZ[1:NX+1, 2:NZ+2] - SZZ[1:NX+1, 1:NZ+1]) * rDZ
    dzsxz = (SXZ[1:NX+1, 1:NZ+1] - SXZ[1:NX+1, 0:NZ  ]) * rDZ

    VX[IN] += (dxsxx + dzsxz) * BX_DT
    VZ[IN] += (dxsxz + dzszz) * BZ_DT

    VX *= G_VX
    VZ *= G_VZ

    # 波形記録
    if it % NSKIP == 0:
        it1 = (it // NSKIP) - 1
        if it1 < NWMAX:
            VWX[it1, :] = VX[ISTX, ISTZ]
            VWZ[it1, :] = VZ[ISTX, ISTZ]

    # 進捗表示とスナップショット出力
    if it % NTDEC == 0:
        vxmax = np.abs(VX).max()
        print(f"{it:5d}/{NTMAX:5d}: T={T:6.2f}[s] vxmax={vxmax:.3e}")
        
        # 空間データ一括計算
        i_idx, k_idx = np.arange(1, NX+1, NXD), np.arange(1, NZ+1, NZD)
        ii, kk = np.meshgrid(i_idx, k_idx, indexing='ij')
        
        # 物理量の計算 (div, rot)
        d_vx_x = (VX[ii, kk] - VX[ii-1, kk]) / DX
        d_vz_z = (VZ[ii, kk] - VZ[ii, kk-1]) / DZ
        d_vx_z = (VX[ii, kk+1] - VX[ii, kk]) / DZ
        d_vz_x = (VZ[ii+1, kk] - VZ[ii, kk]) / DX
        
        out_data = np.column_stack([
            (ii*DX).ravel(), (kk*DZ).ravel(),
            VX[ii, kk].ravel(), VZ[ii, kk].ravel(),
            (d_vx_x + d_vz_z).ravel(), (d_vz_x - d_vx_z).ravel()
        ])
        np.savetxt(f"{ONAME0}{it:05d}.out", out_data, fmt='%9.3f %9.3f %12.3e %12.3e %12.3e %12.3e')

# --- 波形データの最終保存 ---
print("波形データ保存中...")
time_axis = np.arange(1, NWMAX + 1) * DT * NSKIP
for ist in range(NST):
    wfname = f"{WNAME0}{int(ISTX[ist]*DX):04d}.dat"
    np.savetxt(wfname, np.column_stack([time_axis, VWX[:, ist], VWZ[:, ist]]), fmt='%9.3f %14.5e %14.5e')

print("完了")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# --- 設定 ---
fact = 1e4 # scale factor
dt = 0.02
nxs, nzs = NX // NXD, NZ // NZD  # 出力で間引いた格子数

time_val = 16.0
step = int(round(time_val/DT/NTDEC))*NTDEC # 出力間隔NTDECの倍数に丸める
time_val = step*DT
filename = f"{ONAME0}{step:05d}.out"

# データ読み込み
out = pd.read_csv(filename, sep=r'\s+', names=('X','Z','VX','VZ','div','rot'))
    

fig, axs = plt.subplots(2, 2, figsize=(12, 10))
plt.rcParams.update({'font.size': 10})
    
# データ整形
X = out['X'].values.reshape(nxs, nzs)-I0*DX
Z = out['Z'].values.reshape(nxs, nzs)-K0*DZ
vx_s = out['VX'].values.reshape(nxs, nzs)
vz_s = out['VZ'].values.reshape(nxs, nzs)
div_s = out['div'].values.reshape(nxs, nzs)
rot_s = out['rot'].values.reshape(nxs, nzs)
    
# 正規化
vx_s *= fact
vz_s *= fact
div_s *= fact
rot_s *= fact

# 左側：Vx
ax0 = axs[0,0]
im1 = ax0.pcolormesh(X, Z, vx_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_X$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_ylabel('Z from source [km]',fontsize=14)

# 右側：Vz
ax1 = axs[0,1]
im2 = ax1.pcolormesh(X, Z, vz_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$V_Z$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
fig.colorbar(im2, ax=ax1, shrink=0.5).set_ticks([])

# 左側：div v
ax0 = axs[1,0]
im1 = ax0.pcolormesh(X, Z, div_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{div}} \ \mathbf{{v}}$ at {time_val:.2f} s"
ax0.text(0.95, 0.05, label_text,
        transform=ax0.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax0.set_xlabel('X from source [km]',fontsize=14)
ax0.set_ylabel('Z from source [km]',fontsize=14)

# 右側：rot v
ax1 = axs[1,1]
im2 = ax1.pcolormesh(X, Z, rot_s, shading='auto', cmap='RdBu_r', vmin=-1, vmax=1)
label_text = fr"$\mathrm{{rot}} \ \mathbf{{v}}$ at {time_val:.2f} s"
ax1.text(0.95, 0.05, label_text,
        transform=ax1.transAxes,
        fontsize=12, va='bottom', ha='right', fontweight='regular')
ax1.set_xlabel('X from source [km]',fontsize=14)
fig.colorbar(im2, ax=ax1, shrink=0.5).set_ticks([])

for ax in axs.flatten():
    ax.set_xlim(-80, 80)
    ax.set_ylim(-80, 80)
    ax.set_aspect(1.0)
    ax.tick_params(direction="in", top=True, right=True, which='both')

plt.tight_layout()
plt.show()

In [ ]:
import glob

dir_path = OUTDIR  # ファイルがあるディレクトリ
file_pattern = 'wav.h.*.dat'
files = sorted(glob.glob(os.path.join(dir_path, file_pattern)))
dist0 = I0*DX  # 震源の x 座標

# attenuation function
A0 = 0.0016
vs_ref = 3.500

# === プロット ===
plt.rcParams.update({'font.size': 11})
plt.rcParams['xtick.direction'] = 'in' 
plt.rcParams['ytick.direction'] = 'in' 
plt.rcParams["xtick.minor.visible"] = True  
plt.rcParams["ytick.minor.visible"] = True  

plt.figure(figsize=(5, 6))

for file in files:
    data = np.loadtxt(file)
    t  = data[:, 0]
    vz = data[:, 2]
    filename = os.path.basename(file)
    dist = int(filename.split('.')[2])-dist0

    if dist > 0:
        plt.plot(t, vz, label=f"{dist} [km]")
    
A = A0/np.sqrt(t*vs_ref)

plt.plot(t, A, color='black',ls='--',label=r'$1 / \sqrt{r}$')

plt.legend()
plt.xlim(0,30)
plt.ylim(-0.0005, 0.0005)
plt.xlabel("Time [s]", fontsize=14)
plt.title(r"$Q = \infty $")
plt.tight_layout()
plt.show()

In [ ]:
# === プロット ===
plt.rcParams.update({'font.size': 11})
plt.rcParams['xtick.direction'] = 'in' 
plt.rcParams['ytick.direction'] = 'in' 
plt.rcParams["xtick.minor.visible"] = True  
plt.rcParams["ytick.minor.visible"] = True  

plt.figure(figsize=(5, 6))

for file in files:
    data = np.loadtxt(file)
    t  = data[:, 0]
    vz = data[:, 2]
    filename = os.path.basename(file)
    dist = int(filename.split('.')[2])-dist0

    if dist < 0:
        plt.plot(t, -vz, label=f"{dist} [km]")
    
A = A0/np.sqrt(t*vs_ref)

plt.plot(t, A, color='black',ls='--',label=r'$1 / \sqrt{r}$')

plt.legend()
plt.xlim(0,30)
plt.ylim(-0.0005, 0.0005)
plt.xlabel("Time [s]", fontsize=14)
plt.title("$Q = 10$")
plt.tight_layout()
plt.show()